# Trabalho Prático 1 - Aprendizagem Automática
## Previsão de Preços de Carros Usados (Kaggle Competition)

**Licenciatura em Engenharia de Sistemas e Tecnologias Informáticas** 

**Unidade Curricular:** Aprendizagem Automática  
**Ano Letivo:** 2025/2026 

---

### 1. Introdução e Objetivos
Este *notebook* documenta o processo de desenvolvimento de um modelo de regressão avançado para prever o preço de carros usados. O objetivo é minimizar o erro quadrático médio (RMSE) através de uma abordagem focada na qualidade dos dados e na otimização de algoritmos de *gradient boosting*.

A metodologia adotada privilegia o algoritmo **XGBoost** como motor principal de inferência, suportado por uma estratégia de imputação de dados hierárquica e engenharia de atributos robusta.

### 2. Identificação do Grupo
* **Aluno 1:** Bernardo Freitas (79295)
* **Aluno 2:** Afonso Figueiredo (79309)

### 3. Descrição da Abordagem
Para atingir a máxima precisão e simplicidade, optou-se por uma arquitetura baseada num único modelo altamente otimizado, em detrimento de *ensembles* complexos que aumentam o custo computacional sem ganhos significativos de performance neste contexto específico.

**Estratégia Implementada:**
1.  **Imputação Hierárquica:** Preenchimento inteligente de valores em falta (Potência/Cilindrada) utilizando a mediana agrupada por **Marca e Modelo**.
2.  **Pipeline Scikit-Learn:** Encapsulamento de todo o pré-processamento (imputação, normalização `RobustScaler`) para garantir a integridade dos dados durante a validação cruzada.
3.  **XGBoost Otimizado:** Configuração de *shrinkage* (baixa taxa de aprendizagem com elevado número de estimadores) para maximizar a generalização.

### 4. Configuração do Ambiente e Reprodutibilidade

Nesta secção inicializamos as bibliotecas fundamentais para manipulação de dados (`pandas`, `numpy`), visualização (`seaborn`, `matplotlib`) e modelação (`scikit-learn`, `xgboost`).

**Destaques da Configuração:**
* **Reprodutibilidade:** Conforme exigido no enunciado, fixamos as sementes aleatórias (`RANDOM_STATE = 42`) no `numpy`, `python` e `random`. Isto garante que os resultados apresentados no relatório são exatamente replicáveis.
* **Estratégia de Execução (Mode Selection):** Implementou-se uma lógica de alternância entre dois modos:
    * **`quick`:** Para testes rápidos de sanidade do código e *debugging* (menos *folds*, menos iterações).
    * **`full`:** Para a submissão final, utilizando 5 *folds* na validação cruzada e uma busca de hiperparâmetros mais exaustiva.
* **Algoritmo Escolhido:** `XGBRegressor` (XGBoost), que consta na lista de algoritmos permitidos (Random Forest e variantes).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os, joblib, re, random

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor

# Ignorar avisos de depreciação para manter o output limpo na apresentação
warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO GLOBAL ---
MODE = 'full'  # 'quick' para testes, 'full' para produção
TRAIN_N_JOBS = -1 # Utiliza todos os núcleos do processador disponíveis
RANDOM_STATE = 42 # Hitchhiker's Guide to the Galaxy reference (uma inside joke para os fãs de ficção científica)

# Definição de parâmetros baseada no modo de execução
if MODE == 'quick':
    CV_FOLDS = 3
    N_ITER = 2
    XGB_ESTIMATORS = 100
else:
    CV_FOLDS = 5
    N_ITER = 20
    # 4000 árvores permite aprender relações complexas (requer learning_rate baixo pois caso contrário overfitting pode ocorrer)
    XGB_ESTIMATORS = 4000 # muitos passos

os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(f"Ambiente Pronto. Modo: {MODE} | N_Jobs: {TRAIN_N_JOBS}")

### 5. Pré-processamento e Engenharia de Atributos (Feature Engineering)

Nesta etapa, transformamos os dados brutos em *features* utilizáveis pelos modelos. O foco principal foi a correção de inconsistências e a criação de novas variáveis que capturem o valor real de um veículo.

### 5.1. Extração e Limpeza (Regex)
A coluna original `engine` contém texto não estruturado (e.g., *"4.0L V8 Twin Turbo"*). Utilizamos expressões regulares (**Regex**) para extrair três variáveis numéricas fundamentais:
* **HP (Cavalos):** Potência do motor.
* **Liters:** Capacidade volumétrica do motor.
* **Cylinders:** Número de cilindros.

### 5.2. Imputação Hierárquica de Valores em Falta
Para garantir que não introduzimos viés (bias) ao preencher valores nulos na potência e cilindrada, implementámos uma estratégia de **Imputação Hierárquica**. O algoritmo procura o valor mais específico possível:
1.  **Nível 1 (Específico):** Mediana agrupada por **(Marca, Modelo)**. *Ex: Se falta a HP de um 'BMW M3', usamos a mediana dos 'BMW M3'.*
2.  **Nível 2 (Aproximação):** Mediana agrupada por **Marca**. *Ex: Se o modelo é desconhecido, usamos a mediana da 'BMW'.*
3.  **Nível 3 (Genérico):** Mediana **Global** do *dataset*.

### 5.3. Feature Engineering e Encoding
Novas variáveis foram derivadas para ajudar o modelo a capturar relações não lineares:
* `log_milage`: Aplicação de $log(1+x)$ à quilometragem para reduzir a assimetria (skewness) da distribuição.
* `hp_per_liter`: Eficiência do motor ($\frac{HP}{Liters}$).
* `is_super_luxury`: Variável binária para marcas de ultra-luxo, onde a depreciação segue curvas diferentes.
* **Label Encoding:** Conversão de variáveis categóricas em numéricas para processamento pelo XGBoost.

In [ ]:
# --- CARREGAMENTO DE DADOS ---
try:
    # Carrega os datasets de treino e teste
    train_df = pd.read_csv('../data/train.csv')
    test_df = pd.read_csv('../data/test.csv')
except FileNotFoundError:
    print("Erro CRÍTICO: Ficheiros csv não encontrados. Verifique o caminho '../data/'.")

# --- 1. LIMPEZA E EXTRAÇÃO (REGEX) ---
def clean_engine(row):
    """
    Extrai HP, Litros e Cilindros de strings desformatadas (ex: '4.0L V8 Twin Turbo').
    Usa Expressões Regulares (Regex) para capturar padrões numéricos.
    """
    engine = str(row['engine'])
    hp, liters, cylinders = np.nan, np.nan, np.nan
    
    # Extração de HP (Cavalos)
    # Padrão: Digitos (\d+) seguidos opcionalmente por ponto (\.?) e mais digitos, logo antes de 'HP'
    hp_match = re.search(r'(\d+\.?\d*)HP', engine)
    if hp_match: hp = float(hp_match.group(1))
    
    # Extração de Litros (Capacidade)
    # Padrão: Digitos antes de 'L' ou 'Liter'
    lit_match = re.search(r'(\d+\.?\d*)L', engine)
    if not lit_match: lit_match = re.search(r'(\d+\.?\d*) Liter', engine)
    if lit_match: liters = float(lit_match.group(1))
    
    # Extração de Cilindros
    # Padrão: 'V' seguido de digitos (V8, V6) ou digitos seguidos de 'Cylinder'
    cyl_match = re.search(r'V(\d+)', engine)
    if not cyl_match: cyl_match = re.search(r'(\d+) Cylinder', engine)
    if cyl_match: cylinders = float(cyl_match.group(1))
    
    return pd.Series([hp, liters, cylinders])

def map_transmission(trans):
    """
    Reduz a alta cardinalidade da transmissão agrupando em 3 tipos principais.
    Essencial para evitar overfitting em categorias com poucas amostras.
    """
    t = str(trans).lower() # Normaliza para minúsculas
    if 'auto' in t or 'a/t' in t: return 'Automatic' 
    if 'manual' in t or 'm/t' in t: return 'Manual' 
    if 'cvt' in t: return 'CVT'
    return 'Other'

def clean_initial(df):
    data = df.copy()
    
    # Normalização da Quilometragem (Mileage)
    # Remove vírgulas ('12,000' -> '12000') e converte para float
    if 'mileage' in data.columns:
        if data['mileage'].dtype == 'O': # 'O' significa Object (string)
             data['milage'] = data['mileage'].astype(str).str.replace(',', '').str.extract(r'(\d+)')[0].astype(float)
        else:
             data['milage'] = data['mileage']
        
        # Corrige nome da coluna para consistência ('milage' vs 'mileage')
        if 'milage' != 'mileage':
            data.drop(columns=['mileage'], inplace=True, errors='ignore')
    elif 'milage' in data.columns:
        if data['milage'].dtype == 'O':
            data['milage'] = data['milage'].astype(str).str.replace(',', '').str.extract(r'(\d+)')[0].astype(float)
        
    # Aplica a extração de características do motor (cria 3 novas colunas)
    data[['HP', 'Liters', 'Cylinders']] = data.apply(clean_engine, axis=1)
    
    # Engenharia Temporal: Transforma 'Ano' em 'Idade' (Melhor correlação com depreciação)
    data['age'] = 2025 - pd.to_numeric(data['model_year'], errors='coerce')
    
    # Agrupamento da transmissão para reduzir cardinalidade
    data['transmission_grp'] = data['transmission'].apply(map_transmission)
    
    return data

# Aplica limpeza inicial
train_df = clean_initial(train_df)
test_df = clean_initial(test_df)

# --- 2. IMPUTAÇÃO HIERÁRQUICA (Lógica Avançada) ---
print("A aplicar Imputação Hierárquica (Marca+Modelo)...")
# Estratégia: Preencher nulos com a mediana específica do carro, não a média global.

# Criar chave composta para agrupamento (Ex: 'BMW_M3')
train_df['brand_model'] = train_df['brand'].astype(str) + "_" + train_df['model'].astype(str)
test_df['brand_model'] = test_df['brand'].astype(str) + "_" + test_df['model'].astype(str)

# --- PREVENÇÃO DE DATA LEAKAGE ---
# Calculamos as medianas APENAS nos dados de TREINO porque a informação do teste não deve influenciar o treino.
# Se usássemos o teste, estaríamos a "ver o futuro".
map_bm_hp = train_df.groupby('brand_model')['HP'].median() # Mediana por Marca+Modelo
map_b_hp = train_df.groupby('brand')['HP'].median() # Mediana por Marca
global_hp = train_df['HP'].median() # Mediana Global da potencia cavalos

map_bm_lit = train_df.groupby('brand_model')['Liters'].median() # Mediana por Marca+Modelo
map_b_lit = train_df.groupby('brand')['Liters'].median() # Mediana por Marca 
global_lit = train_df['Liters'].median() # Mediana Global dos litros

def hierarchical_fill(row, col, map_bm, map_b, glob):
    """
    Função de preenchimento inteligente com 3 níveis de prioridade.
    """
    if pd.isna(row[col]):
        # Nível 1: Tenta encontrar a mediana do Modelo específico (Marca e Modelo) (Ex: BMW M3)
        val = map_bm.get(row['brand_model'])
        if pd.notna(val): return val
        
        # Nível 2: Se não existe modelo, usa a mediana da Marca (Ex: BMW Geral)
        val = map_b.get(row['brand'])
        if pd.notna(val): return val
        
        # Nível 3: Em último caso, usa a mediana Global do dataset
        return glob
    return row[col]

# Aplica a imputação linha a linha
for df in [train_df, test_df]:
    df['HP'] = df.apply(lambda x: hierarchical_fill(x, 'HP', map_bm_hp, map_b_hp, global_hp), axis=1) 
    df['Liters'] = df.apply(lambda x: hierarchical_fill(x, 'Liters', map_bm_lit, map_b_lit, global_lit), axis=1) 
    
    # Preencher Cilindros e Milhagem com mediana simples (menos críticos)
    for c in ['Cylinders', 'milage']:
        df[c] = df[c].fillna(train_df[c].median())
    
    # --- FEATURE ENGINEERING ---
    # Logaritmo na milhagem: Corrige a distribuição assimétrica (Skewness).
    # Diferença entre 0km e 20k km impacta mais o preço que 100k vs 120k.
    df['log_milage'] = np.log1p(df['milage']) # usei log1p para evitar log(0).
    
    # Rácio de Eficiência (Potência/Litro).
    # replace(0, 1.6) evita divisão por zero.
    df['hp_per_liter'] = df['HP'] / df['Liters'].replace(0, 1.6)
    
    # Marcador de Luxo: Ajuda a árvore de decisão a separar rapidamente carros exóticos
    luxury = ['Bugatti','Lamborghini','Ferrari','McLaren','Rolls-Royce','Bentley','Aston Martin','Porsche','Maserati']
    df['is_super_luxury'] = df['brand'].isin(luxury).astype(int) # 1 se luxo, 0 caso contrário

# --- 3. LABEL ENCODING ---
# Transforma texto em números (ex: BMW -> 1, Audi -> 2).
# Adequado para árvores de decisão (XGBoost) que lidam bem com ordinalidade.
cat_cols = ['brand', 'model', 'fuel_type', 'transmission_grp', 'ext_col', 'int_col', 'accident', 'clean_title']

for col in cat_cols:
    le = LabelEncoder() # Inicializa o encoder
    train_df[col] = train_df[col].astype(str) # Garante que todos os dados são strings
    test_df[col] = test_df[col].astype(str) 
    
    # Fit no conjunto COMPLETO (Treino + Teste)
    # Motivo: Garantir que o Encoder conhece todas as categorias possíveis.
    # Isto NÃO é Data Leakage de Target (não estamos a ver preços), apenas dicionário de strings.
    full_data = pd.concat([train_df[col], test_df[col]], axis=0)
    le.fit(full_data) # Garantir que o encodere conhece todas as categorias possíveis.
    
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

print("Limpeza e Engenharia Concluídas.")

### 6. Pipeline de Modelação e Transformação do Target

Nesta fase, definimos a estrutura de dados que alimentará os modelos. Foram tomadas duas decisões arquiteturais importantes para lidar com a natureza assimétrica dos preços de automóveis:

### 6.1. Transformação Logarítmica do Target
A variável alvo (`price`) possui uma assimetria positiva (cauda longa à direita). Para estabilizar a variância e melhorar a convergência do algoritmo, aplicámos uma transformação logarítmica:
$$y = \log(1 + \text{price})$$
Isto garante que:
1.  O modelo não prevê preços negativos.
2.  A métrica de erro penaliza desvios relativos (erros percentuais) em vez de absolutos, o que é crucial dado que o dataset contém desde carros baratos a supercarros de luxo.

### 6.2. Pipeline de Preprocessamento
Utilizámos um `ColumnTransformer` para garantir que todas as transformações são aplicadas de forma isolada, evitando *Data Leakage*.
* **Normalização:** Aplicámos `RobustScaler` a todas as *features*. Embora o XGBoost seja invariante à escala, esta normalização é fundamental para que outros modelos exigidos no enunciado (KNN, SVM, Redes Neuronais) funcionem corretamente, permitindo a comparação direta entre algoritmos.
* **Safety Net:** Um `SimpleImputer` foi incluído no pipeline para tratar eventuais valores nulos residuais que tenham escapado à imputação hierárquica.

In [ ]:
# --- DEFINIÇÃO DE FEATURES ---
features = ['brand', 'model', 'age', 'log_milage', 'HP', 'Liters', 'Cylinders', 'hp_per_liter',
            'fuel_type', 'transmission_grp', 'ext_col', 'int_col', 'accident', 'clean_title', 'is_super_luxury']

X = train_df[features] 
y = np.log1p(train_df['price']) # Log-transform no target para estabilizar variância 

# O RobustScaler é aplicado a TODAS as features. Justificativa: Embora redundante para Árvores (XGBoost), é OBRIGATÓRIO para modelos baseados em distância (KNN, SVM).
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')), # Imputa valores faltantes com a mediana
            ('scaler', RobustScaler()) # Robusto a outliers (essencial para preços e km)
        ]), features)
    ])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE) 
print("Split de dados concluído.")

### 7. Treino e Otimização de Hiperparâmetros (XGBoost)

Para o modelo final, selecionámos o **XGBoost (Extreme Gradient Boosting)** devido à sua capacidade de lidar com dados tabulares complexos e à sua implementação eficiente de *Gradient Boosted Decision Trees*.

### 7.1. Estratégia de Procura (RandomizedSearchCV)
Em vez de uma *Grid Search* exaustiva (que seria computacionalmente proibitiva com tantas variáveis), optámos por uma **Randomized Search** com validação cruzada ($k=5$). Isto permite explorar uma área maior do espaço de hiperparâmetros de forma eficiente.

### 7.2. Definição do Espaço de Hiperparâmetros
A grelha de pesquisa foi desenhada para controlar o equilíbrio entre viés e variância (*bias-variance tradeoff*):
* **Capacidade de Aprendizagem:**
    * `learning_rate` ($0.007 - 0.01$): Taxas muito baixas exigem mais árvores (`n_estimators`), mas garantem uma convergência mais robusta ao mínimo global da função de perda.
* **Controlo de Overfitting (Regularização):**
    * `max_depth`: Limita a complexidade de cada árvore individual.
    * `reg_alpha` (L1) e `reg_lambda` (L2): Termos de penalização na função objetivo para evitar pesos excessivos nas folhas.
* **Estocasticidade (Stochastic Boosting):**
    * `subsample` e `colsample_bytree`: Treinam cada árvore com apenas uma fração das amostras e das *features*, respetivamente. Isto descorrelaciona as árvores e reduz a variância do modelo final.

### 7.3. Métrica de Avaliação
Utilizámos o `neg_mean_squared_error` como função de pontuação interna, pois o Scikit-Learn segue a convenção de "maximizar" a pontuação (daí o sinal negativo). O erro final reportado será a **RMSE** na escala real do preço (revertendo o logaritmo).

In [ ]:
# 1. Definição do Pipeline do Modelo
# O pipeline garante que o pré-processamento é aplicado dentro de cada fold da validação cruzada
xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(objective='reg:squarederror', n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE))]) # reg:squarederror para regressão

# 2. Grelha de Hiperparâmetros
# Focada em regularização para evitar que o modelo memorize o treino (overfitting)
xgb_params = {
    'model__n_estimators': [XGB_ESTIMATORS], # Definido na config global (4000 para modo 'full')
    'model__learning_rate': [0.007, 0.01], # Learning rate baixo = melhor generalização (passos pequenos)
    'model__max_depth': [6, 8, 10], # Profundidade da árvore (6 por padrão, 8 e 10 para mais complexidade)
    'model__subsample': [0.65, 0.75], # Fração de linhas usadas por árvore
    'model__colsample_bytree': [0.65, 0.75], # Fração de colunas usadas por árvore
    'model__reg_alpha': [0.1, 0.5], # Regularização L1 (Lasso)
    'model__reg_lambda': [1.0, 2.0] # Regularização L2 (Ridge)
}

# 3. PROCESSO DE TREINO (Cross-Validation)
print("A iniciar otimização do XGBoost (isto pode demorar alguns minutos)...")
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(xgb_pipe, xgb_params, n_iter=N_ITER, cv=cv, # Usar K-Fold Cross-Validation com shuffling para melhor generalização 
                            scoring='neg_mean_squared_error', n_jobs=TRAIN_N_JOBS, # O neg_ serve apenas para enganar o algoritmo para ele achar que está a subir uma montanha (score), quando na verdade está a descer um buraco (erro).
                            random_state=RANDOM_STATE, verbose=1)

search.fit(X_train, y_train)
best_model = search.best_estimator_

# 4. Avaliação Final
preds = best_model.predict(X_val) # Previsão nos dados de validação (que o modelo nunca viu durante o treino)
rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(preds))) # Reverte o log1p para calcular RMSE no espaço original
print(f"\n>>> MELHOR RMSE VALIDAÇÃO: {rmse:,.2f}")
print(f"Melhores Parâmetros: {search.best_params_}")

# Guardar modelo
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/XGB_Final_Optimized.joblib')

### 8. Submissão
Geração das previsões finais para o conjunto de teste, revertendo a transformação logarítmica (`expm1`) e garantindo que não existem valores negativos.

In [ ]:
print("A gerar submissão...")

# 1. Prever Teste
X_test_final = test_df[features]

# Faz a previsão com o melhor modelo treinado
log_preds = best_model.predict(X_test_final)

# Reverte a transformação logarítmica (expm1 é o inverso de log1p)
final_preds = np.expm1(log_preds)

# 2. Guardar ficheiro
# Cria o DataFrame final, garantindo que não existem preços negativos (clip em 0)
submission = pd.DataFrame({'id': test_df['id'], 'price': np.clip(final_preds, 0, None)})
submission.to_csv('submission.csv', index=False)

print("Ficheiro 'submission.csv' gerado com sucesso!")

### 9. Análise Visual e Diagnóstico de Desempenho Preditivo

Nesta etapa, realizamos a inferência final no conjunto de validação e procedemos à avaliação qualitativa do modelo. Dado que o treino foi realizado no espaço logarítmico (para normalizar a distribuição enviesada dos preços), é imperativo reverter essa transformação para interpretar os resultados na escala monetária real (€).

A transformação inversa é aplicada através da função exponencial:
$$\hat{y}_{eur} = e^{\hat{y}_{log}} - 1$$

### Interpretação dos Painéis Gráficos

A visualização tripartida permite um diagnóstico completo do comportamento do modelo:

1.  **Distribuição de Preços (Esquerda):**
    * Analisa a densidade de probabilidade das previsões no conjunto de teste.
    * Permite verificar se o modelo replicou a distribuição "cauda longa" (*long-tail*) típica de mercados imobiliários/financeiros, ou se apresenta viés em direção à média.

2.  **Precisão Global e Outliers (Centro):**
    * Gráfico de dispersão (*scatter plot*) comparando os valores reais ($y$) vs. previstos ($\hat{y}$) em toda a amplitude dos dados.
    * A linha vermelha tracejada representa o **ajuste ideal** ($y = \hat{y}$). Pontos distantes desta linha indicam erros residuais significativos, permitindo identificar visualmente a presença de *outliers* extremos que podem distorcer métricas como o RMSE.

3.  **Zoom de Precisão - 99% dos Dados (Direita):**
    * Foca a análise na grande maioria das amostras, excluindo o 1% superior (imóveis de luxo extremo ou anomalias de dados).
    * Esta visualização é crucial para validar a **homocedasticidade** (se a variância do erro se mantém constante) e a precisão do modelo no segmento de mercado mais frequente.

In [ ]:
# --- VISUALIZAÇÃO E ANÁLISE ---
val_predictions = best_model.predict(X_val)
y_real = np.expm1(y_val.values) if hasattr(y_val, 'values') else np.expm1(y_val)
y_pred_val = np.expm1(val_predictions)
df_results = pd.DataFrame({'Real': y_real, 'Previsto': y_pred_val})
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
sns.set_theme(style="whitegrid")

# === GRÁFICO 1: Distribuição de Preços (Submissão/Teste) ===
sns.histplot(final_preds, bins=60, kde=True, color='#2c3e50', line_kws={'linewidth': 2}, alpha=0.7, ax=axes[0])
mediana = np.median(final_preds)
axes[0].axvline(mediana, color='#e74c3c', linestyle='--', linewidth=2, label=f'Mediana: €{mediana:,.0f}')
axes[0].set_title('Distribuição de Preços (Conjunto de Teste)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Preço Previsto (€)', fontsize=12)
axes[0].set_ylabel('Frequência', fontsize=12)
axes[0].legend()

# === GRÁFICO 2: Visão Geral Real vs Previsto (Validação) ===
sns.scatterplot(data=df_results, x='Real', y='Previsto', alpha=0.5, color='#2980b9', ax=axes[1])
max_val = max(df_results['Real'].max(), df_results['Previsto'].max())
axes[1].plot([0, max_val], [0, max_val], '--r', linewidth=2, label='Ideal')
axes[1].set_title('Precisão Geral (Com Outliers)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Preço Real (€)', fontsize=12)
axes[1].set_ylabel('Preço Previsto (€)', fontsize=12)
axes[1].legend()

# === GRÁFICO 3: Zoom 99% (Validação) ===
limite_visual = np.percentile(y_real, 99) 

sns.scatterplot(data=df_results, x='Real', y='Previsto', alpha=0.5, color='#2980b9', ax=axes[2])
axes[2].plot([0, limite_visual], [0, limite_visual], '--r', linewidth=2, label='Ideal')

# Forçar limites
axes[2].set_xlim(0, limite_visual)
axes[2].set_ylim(0, limite_visual)
axes[2].set_title(f'Zoom de Precisão (99% dos dados)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Preço Real (€)', fontsize=12)
axes[2].set_ylabel('Preço Previsto (€)', fontsize=12)
axes[2].legend()

plt.tight_layout()
plt.show()

### 10. Análise Comparativa e Justificação Teórica dos Modelos

Conforme solicitado nos objetivos do trabalho prático, a seleção do **XGBoost** como modelo final não foi arbitrária. Para garantir rigor científico, analisámos o comportamento teórico e prático de diversos algoritmos listados no enunciado, comparando as suas limitações face à natureza deste *dataset* (previsão de preços de automóveis).

### 10.1. Regressão Linear (Linear Regression)
* **Funcionamento:** Tenta modelar a relação entre as variáveis (idade, potência) e o preço através de uma equação linear simples ($y = mx + b$).
* **Por que não foi escolhido?**
    * **Alta de Linearidade:** O algoritmo assume que a depreciação de um carro é constante. Na realidade, a desvalorização é não-linear (exponencial nos primeiros anos, estagnando em clássicos), o que leva a um elevado **Viés (Underfitting)**.

### 10.2. K-Nearest Neighbors (KNN)
* **Funcionamento:** Algoritmo baseado em distância. Para prever o valor de um carro, procura os "K" carros mais parecidos no histórico e faz a média dos seus preços.
* **Por que não foi escolhido?**
    * **Maldição da Dimensionalidade:** Com muitas variáveis (especialmente após o *One-Hot/Label Encoding* das marcas e modelos), o espaço torna-se muito disperso, fazendo com que o cálculo de distância perca eficácia.
    * **Sensibilidade a Ruído:** É muito afetado por *outliers* (carros com preços anormais), comuns neste *dataset*.

### 10.3. Árvores de Decisão (Decision Trees)
* **Funcionamento:** Cria regras de "se-então" (ex: *Se Idade > 5 anos e Marca = BMW...*) para dividir os dados em grupos mais puros.
* **Por que não foi escolhido?**
    * **Alta Variância (Overfitting):** Uma única árvore tende a "decorar" os dados de treino. Pequenas alterações no *dataset* (ruído) geram árvores completamente diferentes, resultando em má generalização para novos dados.

### 10.4. Random Forest
* **Funcionamento:** Técnica de *Bagging*. Treina centenas de árvores de decisão independentes em paralelo e faz a média das suas previsões para reduzir a variância.
* **Comparação com a Escolha Final:**
    * Embora seja um excelente modelo, o Random Forest foca-se na redução da **variância**. O XGBoost, sendo um método de *Boosting*, foca-se na redução do **viés** e da variância sequencialmente, obtendo tipicamente erros menores (RMSE) em dados tabulares estruturados.

Para ilustrar a razão da escolha do XGBoost sobre o Random Forest, considere-se o seguinte esquema comparativo:

<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*6n0rqLRc8ljOsNlQ2Z5xug.png" width="800" alt="Comparação Bagging vs Boosting">
  <p><em>Figura 1: Diferença estrutural. O Random Forest (Bagging) cria árvores em paralelo (independentes). O XGBoost (Boosting) cria árvores sequenciais, corrigindo os erros anteriores.</em></p>
</div>

Analisando a **Figura 1**, destacam-se as diferenças fundamentais:

1.  **Bagging (Esquerda):** As árvores são construídas de forma **paralela**. O foco é reduzir a variância (estabilidade) através da média das previsões.
2.  **Boosting (Direita):** O processo é **sequencial**. Cada novo modelo (Model 02, 03) recebe o input do anterior. O XGBoost foca-se nos casos onde o modelo anterior falhou (os resíduos), reduzindo o **viés** e aumentando a precisão do preço final.

### 10.5. Support Vector Machines (SVM - SVR)
* **Funcionamento:** Tenta encontrar um hiperplano num espaço dimensional superior que se ajuste aos dados dentro de uma margem de erro.
* **Por que não foi escolhido?**
    * **Escalabilidade:** A complexidade computacional do SVM cresce drasticamente com o número de amostras (perto de $O(n^3)$). Para um *dataset* com dezenas de milhares de linhas, o treino é proibitivamente lento em comparação com métodos de árvores.

### 10.6. Redes Neuronais (Neural Networks / MLP)
* **Funcionamento:** Utilizam camadas de neurónios artificiais para aprender relações complexas não-lineares através de *Backpropagation*.
* **Por que não foi escolhido?**
    * **Adequação aos Dados:** Redes Neuronais brilham em dados não estruturados (imagens, áudio). Para dados tabulares (Excel/CSV) de tamanho médio, algoritmos de *Gradient Boosting* (como XGBoost) são o estado da arte (*State-of-the-Art*), oferecendo performance superior com menos necessidade de ajuste fino de arquitetura.
    
---

### 10.7. Conclusão: A Escolha do XGBoost
O **XGBoost (Extreme Gradient Boosting)** foi selecionado por oferecer o melhor equilíbrio:
1.  **Gradient Boosting:** Corrige iterativamente os erros dos modelos anteriores.
2.  **Regularização (L1/L2):** Previne o *overfitting* melhor que as árvores simples.
3.  **Gestão de Nulos:** Lida nativamente com dados em falta, aprendendo a melhor direção para os tratar.